In [18]:
# Cell 1: imports, paths, and diagnostic settings

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import wasserstein_distance


TASK_DIR = Path.cwd().parent
INPUT_DIR = TASK_DIR / "input"
OUTPUT_DIR = TASK_DIR / "output"

LONG_PATH = INPUT_DIR / "productivity_long.csv"
TRANSITIONS_PATH = INPUT_DIR / "productivity_transitions.csv"
EMP_TRAJECTORIES_PATH = INPUT_DIR / "trajectories_complete.npy"
SIM_TRAJECTORIES_PATH = INPUT_DIR / "simulated_trajectories.npy"

CANONICAL_PATH = OUTPUT_DIR / "canonical_trajectory_stats.csv"
YEARWISE_PATH = OUTPUT_DIR / "yearwise_productivity_stats.csv"
LOG_DELTA_PATH = OUTPUT_DIR / "yearwise_log_delta_stats.csv"
YEAR_MAX_PATH = OUTPUT_DIR / "year_of_maximum.csv"
CUM_Y5_PATH = OUTPUT_DIR / "cumulative_through_year5.csv"
KS_PATH = OUTPUT_DIR / "ks_diagnostics.csv"
LAPLACE_PATH = OUTPUT_DIR / "stage_raw_increment_laplace.csv"
LAPLACE_VALUES_PATH = OUTPUT_DIR / "stage_raw_increment_values.csv"
AGGREGATE_PATH = OUTPUT_DIR / "aggregate_productivity_values.csv"
LOGNORMAL_FIT_PATH = OUTPUT_DIR / "lognormal_fit_stats.csv"
QQ_PATH = OUTPUT_DIR / "lognormal_qq_coordinates.csv"
RANK_CORR_PATH = OUTPUT_DIR / "rank_correlations.csv"
RANK_DECILE_PATH = OUTPUT_DIR / "rank_decile_transitions.csv"
RANK_CHANGE_PATH = OUTPUT_DIR / "rank_change_matrices.npz"
WASSERSTEIN_PATH = OUTPUT_DIR / "wasserstein_by_year.csv"
INEQUALITY_PATH = OUTPUT_DIR / "inequality_summary.csv"
SUMMARY_PATH = OUTPUT_DIR / "diagnostic_summary.csv"

MODEL_NAME = "Stagewise AR(1)-GRW"
MODEL_TAG = "stagewise_ar1_grw"

Y = 20
YEARS = np.arange(Y + 1)
TRANSITION_YEARS = np.arange(Y)
RANK_WINDOWS = [(0, 5),(0, 10),(0, 20),(10, 15),(10, 20),(15, 20)]

STAGE_TRANSITIONS = [
    {"stage": "years_1_4","transition_start": 0,"transition_end": 3},
    {"stage": "years_5_7","transition_start": 4,"transition_end": 6},
    {"stage": "years_8_20","transition_start": 7,"transition_end": 19,}]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



In [19]:


productivity_long = pd.read_csv(LONG_PATH,dtype={"dblp_id": "string", "dblp": "string"})

productivity_transitions = pd.read_csv(TRANSITIONS_PATH,dtype={"dblp_id": "string", "dblp": "string"})

emp_trajectories = np.load(EMP_TRAJECTORIES_PATH)
sim_trajectories = np.load(SIM_TRAJECTORIES_PATH)



EPS = 0.49

assert emp_trajectories.ndim == 2
assert sim_trajectories.ndim == 2


assert np.isfinite(emp_trajectories).all()
assert np.isfinite(sim_trajectories).all()
assert (emp_trajectories >= 0).all()
assert (sim_trajectories >= 0).all()

assert productivity_transitions["CareerAge_next"].eq(productivity_transitions["CareerAge"] + 1).all()


In [20]:

def binned_mode(values, bins="auto"):
    values = pd.Series(values).replace([np.inf, -np.inf],np.nan).dropna().to_numpy(dtype=float)

    if len(values) == 0:
        return np.nan

    if values.min() == values.max():
        return float(values[0])

    counts, edges = np.histogram(values, bins=bins)
    index = int(np.argmax(counts))

    return float((edges[index] + edges[index + 1]) / 2)


def yearwise_stats_from_long(df, eps):
    rows = []

    for year in YEARS:
        q = (df.loc[df["CareerAge"].eq(year), "pubs_adj"].replace([np.inf, -np.inf], np.nan).dropna().astype(float))

        log_q = np.log(q + eps)
        positive_log_q = np.log(q[q > 0])

        rows.append({
        "source": "empirical",
        "career_age": int(year),
        "n": int(len(q)),
        "frac_zero": float((q == 0).mean()),
        "mean_prod": float(q.mean()),
        "median_prod": float(q.median()),
        "sd_prod": float(q.std(ddof=0)),
        "mean_log_prod": float(log_q.mean()),
        "var_log_prod": float(log_q.var(ddof=0)),
        "mean_log_prod_pos": float(positive_log_q.mean()),
        "var_log_prod_pos": float(positive_log_q.var(ddof=0)),
        "q25_prod": float(q.quantile(0.25)),
        "q50_prod": float(q.quantile(0.50)),
        "q75_prod": float(q.quantile(0.75)),
        "q90_prod": float(q.quantile(0.90)),
        "q95_prod": float(q.quantile(0.95))})

    return pd.DataFrame(rows)


def yearwise_stats_from_trajectories(trajectories, eps):
    rows = []

    for year in YEARS:
        q = pd.Series(trajectories[year], dtype=float)
        log_q = np.log(q + eps)
        positive_log_q = np.log(q[q > 0])

        rows.append({
        "source": "simulated",
        "career_age": int(year),
        "n": int(len(q)),
        "frac_zero": float((q == 0).mean()),
        "mean_prod": float(q.mean()),
        "median_prod": float(q.median()),
        "sd_prod": float(q.std(ddof=0)),
        "mean_log_prod": float(log_q.mean()),
        "var_log_prod": float(log_q.var(ddof=0)),
        "mean_log_prod_pos": float(positive_log_q.mean()),
        "var_log_prod_pos": float(positive_log_q.var(ddof=0)),
        "q25_prod": float(q.quantile(0.25)),
        "q50_prod": float(q.quantile(0.50)),
        "q75_prod": float(q.quantile(0.75)),
        "q90_prod": float(q.quantile(0.90)),
        "q95_prod": float(q.quantile(0.95))})

    return pd.DataFrame(rows)


def summarize_log_deltas(delta_matrix, source):
    rows = []

    for year in TRANSITION_YEARS:
        values = pd.Series(delta_matrix[year],dtype=float).replace([np.inf, -np.inf], np.nan).dropna()

        rows.append({
        "source": source,
        "transition_year": int(year),
        "destination_year": int(year + 1),
        "n": int(len(values)),
        "mean_log_delta": float(values.mean()),
        "median_log_delta": float(values.median()),
        "mode_log_delta": binned_mode(values),
        "var_log_delta": float(values.var(ddof=0)),
        "q25_log_delta": float(values.quantile(0.25)),
        "q50_log_delta": float(values.quantile(0.50)),
        "q75_log_delta": float(values.quantile(0.75)),
        "q90_log_delta": float(values.quantile(0.90)),
        "q95_log_delta": float(values.quantile(0.95))})

    return pd.DataFrame(rows)


def laplace_summary(values, source, stage):
    values = pd.Series(values).replace([np.inf, -np.inf],np.nan).dropna().to_numpy(dtype=float)

    mu = float(np.median(values))
    alpha = float(np.mean(np.abs(values - mu)))

    if len(values) == 0 or alpha <= 0:
        ks_stat = np.nan
        ks_pvalue = np.nan
    else:
        ks_result = stats.kstest(values,"laplace",args=(mu, alpha))
        ks_stat = float(ks_result.statistic)
        ks_pvalue = float(ks_result.pvalue)

    return {
    "source": source,
    "stage": stage,
    "n": int(len(values)),
    "mu_hat": mu,
    "alpha_hat": alpha,
    "ks_stat": ks_stat,
    "ks_pvalue": ks_pvalue,
    "mean": float(np.mean(values)),
    "sd": float(np.std(values, ddof=0)),
    "q01": float(np.quantile(values, 0.01)),
    "q50": float(np.quantile(values, 0.50)),
    "q99": float(np.quantile(values, 0.99))}


def fit_lognormal(values, source, aggregation, eps):
    values = np.asarray(values, dtype=float)
    shifted = values + eps

    assert np.isfinite(shifted).all()
    assert (shifted > 0).all()

    shape, loc, scale = stats.lognorm.fit(shifted,floc=0)

    ks_result = stats.kstest(shifted,stats.lognorm.cdf,args=(shape, loc, scale))

    theoretical, observed = stats.probplot(shifted,dist=stats.lognorm,sparams=(shape, loc, scale),fit=False)

    fit_row = {
    "source": source,
    "aggregation": aggregation,
    "n": int(len(values)),
    "epsilon_shift": float(eps),
    "shape": float(shape),"loc": float(loc),"scale": float(scale),"ks_stat": float(ks_result.statistic),"ks_pvalue": float(ks_result.pvalue)}

    qq = pd.DataFrame({
    "source": source,
    "aggregation": aggregation,
    "quantile_index": np.arange(len(values), dtype=int),
    "theoretical_quantile": np.asarray(theoretical, dtype=float),
    "observed_quantile": np.asarray(observed, dtype=float)})

    return fit_row, qq


def percentile_ranks(values):
    return pd.Series(values).rank(method="average",pct=True).to_numpy()




In [21]:

emp_yearwise = yearwise_stats_from_long(productivity_long,EPS)

sim_yearwise = yearwise_stats_from_trajectories(sim_trajectories,EPS)

yearwise_productivity_stats = pd.concat([emp_yearwise, sim_yearwise],ignore_index=True)

canonical_trajectory_stats = (yearwise_productivity_stats[
    ["source",
    "career_age",
    "n",
    "mean_prod",
    "median_prod",]].rename(columns={"mean_prod": "mean","median_prod": "median"}))

display(canonical_trajectory_stats.head())
display(yearwise_productivity_stats.head())

,source,career_age,n,mean,median
0,empirical,0,2032,4.318336,3.225469
1,empirical,1,2112,4.632843,3.345619
2,empirical,2,2172,5.888196,4.667694
3,empirical,3,2213,7.046556,5.549047
4,empirical,4,2244,7.517097,5.988513


,source,career_age,n,frac_zero,mean_prod,median_prod,sd_prod,mean_log_prod,var_log_prod,mean_log_prod_pos,var_log_prod_pos,q25_prod,q50_prod,q75_prod,q90_prod,q95_prod
0,empirical,0,2032,0.210138,4.318336,3.225469,4.680764,1.097790,1.173361,1.448587,0.478700,1.451031,3.225469,6.095144,9.676407,12.126249
1,empirical,1,2112,0.201705,4.632843,3.345619,4.990201,1.145552,1.209594,1.486163,0.528669,1.451031,3.345619,6.352937,10.220042,13.983260
2,empirical,2,2172,0.145028,5.888196,4.667694,5.458139,1.421408,1.140577,1.673712,0.530436,2.117646,4.667694,8.323571,12.447185,15.616860
3,empirical,3,2213,0.115228,7.046556,5.549047,6.489896,1.597363,1.115968,1.798668,0.575310,2.625676,5.549047,9.770540,14.922443,18.799346
4,empirical,4,2244,0.114528,7.517097,5.988513,6.811562,1.654971,1.149682,1.867075,0.578746,2.805193,5.988513,10.429070,15.621655,20.110295


In [22]:

emp_log_delta = (np.log(productivity_transitions["pubs_adj_next"].to_numpy(dtype=float)+ EPS) - np.log(productivity_transitions["pubs_adj"].to_numpy(dtype=float)+ EPS))

emp_delta_table = productivity_transitions[["CareerAge"]].copy()

emp_delta_table["log_delta"] = emp_log_delta

emp_delta_matrix = [emp_delta_table.loc[emp_delta_table["CareerAge"].eq(year),"log_delta"].to_numpy(dtype=float) for year in TRANSITION_YEARS]

sim_log_trajectories = np.log(sim_trajectories + EPS)
sim_log_delta_matrix = np.diff(sim_log_trajectories,axis=0)

emp_log_delta_stats = summarize_log_deltas(emp_delta_matrix,"empirical")

sim_log_delta_stats = summarize_log_deltas(sim_log_delta_matrix,"simulated")

yearwise_log_delta_stats = pd.concat([emp_log_delta_stats, sim_log_delta_stats],ignore_index=True)

display(yearwise_log_delta_stats.head())

,source,transition_year,destination_year,n,mean_log_delta,median_log_delta,mode_log_delta,var_log_delta,q25_log_delta,q50_log_delta,q75_log_delta,q90_log_delta,q95_log_delta
0,empirical,0,1,2030,0.028679,-0.022202,-0.112290,1.592889,-0.582398,-0.022202,0.604349,1.682465,2.263652
1,empirical,1,2,2107,0.267982,0.173528,-0.014094,1.435240,-0.245302,0.173528,0.880400,1.899316,2.416097
2,empirical,2,3,2166,0.172183,0.047782,-0.001640,1.172881,-0.308019,0.047782,0.623639,1.437439,2.228056
3,empirical,3,4,2208,0.057367,-0.022202,-0.024154,1.105191,-0.413047,-0.022202,0.564610,1.211450,1.966633
4,empirical,4,5,2221,0.024490,-0.025270,-0.026085,1.085649,-0.416493,-0.025270,0.499858,1.151214,1.934788


In [23]:

emp_year_max = np.argmax(emp_trajectories,axis=0)

sim_year_max = np.argmax(sim_trajectories,axis=0)

emp_cum_y5 = emp_trajectories[:6].sum(axis=0)
sim_cum_y5 = sim_trajectories[:6].sum(axis=0)

year_of_maximum = pd.concat([pd.DataFrame({
    "source": "empirical",
    "sample_index": np.arange(len(emp_year_max)),
    "year_of_maximum": emp_year_max}),
    pd.DataFrame({"source": "simulated",
        "sample_index": np.arange(len(sim_year_max)),
        "year_of_maximum": sim_year_max})],ignore_index=True)

cumulative_through_year5 = pd.concat([pd.DataFrame({
    "source": "empirical",
    "sample_index": np.arange(len(emp_cum_y5)),
    "cumulative_productivity": emp_cum_y5}),
    pd.DataFrame({"source": "simulated",
    "sample_index": np.arange(len(sim_cum_y5)),
    "cumulative_productivity": sim_cum_y5})],ignore_index=True)

ks_year_max = stats.ks_2samp(emp_year_max,sim_year_max)

ks_cum_y5 = stats.ks_2samp(emp_cum_y5,sim_cum_y5)

ks_diagnostics = pd.DataFrame([
    {"diagnostic": "year_of_maximum",
    "statistic": float(ks_year_max.statistic),
    "pvalue": float(ks_year_max.pvalue),
    "n_empirical": int(len(emp_year_max)),
    "n_simulated": int(len(sim_year_max))},
    {"diagnostic": "cumulative_through_year5",
    "statistic": float(ks_cum_y5.statistic),
    "pvalue": float(ks_cum_y5.pvalue),
    "n_empirical": int(len(emp_cum_y5)),
    "n_simulated": int(len(sim_cum_y5))}])

display(ks_diagnostics)

,diagnostic,statistic,pvalue,n_empirical,n_simulated
0,year_of_maximum,0.069540,0.00398,681,10000
1,cumulative_through_year5,0.044716,0.15125,681,10000


In [24]:

emp_raw_deltas = productivity_transitions["raw_delta"].to_numpy(dtype=float)

emp_transition_years = productivity_transitions["CareerAge"].to_numpy(dtype=int)

sim_raw_delta_matrix = np.diff(sim_trajectories,axis=0)

laplace_rows = []
laplace_value_frames = []

for spec in STAGE_TRANSITIONS:
    stage = spec["stage"]
    start = spec["transition_start"]
    end = spec["transition_end"]

    emp_values = emp_raw_deltas[(emp_transition_years >= start)& (emp_transition_years <= end)]

    sim_values = sim_raw_delta_matrix[start:end + 1].ravel()

    laplace_rows.append(laplace_summary(emp_values,"empirical",stage))

    laplace_rows.append(laplace_summary(sim_values,"simulated",stage))

    laplace_value_frames.extend([
        pd.DataFrame({"source": "empirical","stage": stage,"value": emp_values}),
        pd.DataFrame({"source": "simulated","stage": stage,"value": sim_values})])

stage_raw_increment_laplace = pd.DataFrame(laplace_rows)

stage_raw_increment_values = pd.concat(laplace_value_frames,ignore_index=True)

display(stage_raw_increment_laplace)


,source,stage,n,mu_hat,alpha_hat,ks_stat,ks_pvalue,mean,sd,q01,q50,q99
0,empirical,years_1_4,8511,0.000000,3.704202,0.112877,6.479122e-95,0.797559,5.242790,-12.088957,0.000000,15.424012
1,simulated,years_1_4,40000,0.213398,6.406644,0.073538,1.468575e-188,0.882932,13.340955,-32.677446,0.213398,41.569030
2,empirical,years_5_7,6486,-0.134692,4.019228,0.074795,5.303768e-32,-0.217804,5.685402,-15.360186,-0.134692,15.286262
3,simulated,years_5_7,30000,0.062624,6.965179,0.070914,1.245726e-131,0.172049,14.401958,-40.135594,0.062624,42.231313
4,empirical,years_8_20,19112,-0.091652,3.414527,0.103124,2.058493e-177,-0.244873,4.958868,-14.006611,-0.091652,13.832535
5,simulated,years_8_20,130000,-0.015854,5.724676,0.080077,0.000000e+00,-0.136804,12.280169,-35.839518,-0.015854,34.875367


In [25]:
# Cell 8: aggregate productivity distributions

emp_last_year = (
    productivity_long
    .sort_values(["dblp_id", "CareerAge"])
    .groupby("dblp_id", as_index=False)
    .agg(
        value=("pubs_adj", "last"),
        end_career_age=("CareerAge", "last"),
    )
)

emp_last_four = (
    productivity_long
    .sort_values(["dblp_id", "CareerAge"])
    .assign(
        rolling_four=lambda d:
            d.groupby("dblp_id")["pubs_adj"]
             .rolling(4, min_periods=4)
             .sum()
             .reset_index(level=0, drop=True)
    )
    .dropna(subset=["rolling_four"])
    .groupby("dblp_id", as_index=False)
    .agg(
        value=("rolling_four", "last"),
        end_career_age=("CareerAge", "last"),
    )
)

aggregate_arrays = {
    ("empirical", "last_year"): emp_last_year["value"].to_numpy(),
    ("simulated", "last_year"): sim_trajectories[-1],
    ("empirical", "last_four"): emp_last_four["value"].to_numpy(),
    ("simulated", "last_four"): sim_trajectories[-4:].sum(axis=0),
    ("empirical", "full_career"): emp_trajectories.sum(axis=0),
    ("simulated", "full_career"): sim_trajectories.sum(axis=0),
}

aggregate_frames = []

for (source, aggregation), values in aggregate_arrays.items():
    frame = pd.DataFrame({
        "source": source,
        "aggregation": aggregation,
        "sample_index": np.arange(len(values)),
        "value": np.asarray(values, dtype=float),
    })

    if source == "empirical" and aggregation == "last_year":
        frame["end_career_age"] = emp_last_year["end_career_age"].to_numpy()

    elif source == "empirical" and aggregation == "last_four":
        frame["end_career_age"] = emp_last_four["end_career_age"].to_numpy()

    elif source == "simulated" and aggregation in {"last_year", "last_four"}:
        frame["end_career_age"] = Y

    else:
        frame["end_career_age"] = np.nan

    aggregate_frames.append(frame)

aggregate_productivity_values = pd.concat(
    aggregate_frames,
    ignore_index=True,
)

display(
    aggregate_productivity_values.groupby(
        ["source", "aggregation"]
    )["value"].agg(["count", "mean", "median", "max"])
)

count        mean      median          max
source    aggregation                                            
empirical full_career    681  149.054211  120.036692   775.677058
          last_four     2333   22.512645   17.142163   226.473847
          last_year     2375    4.754703    3.176727    51.200687
simulated full_career  10000  148.105016  128.316384  1334.673698
          last_four    10000   26.423956   17.049964   408.602606
          last_year    10000    6.601790    3.245297   200.892698

In [26]:
lognormal_fit_rows = []
qq_frames = []

for (source, aggregation), values in aggregate_arrays.items():
    fit_row, qq_frame = fit_lognormal(values,source,aggregation,EPS)
    lognormal_fit_rows.append(fit_row)
    qq_frames.append(qq_frame)

lognormal_fit_stats = pd.DataFrame(lognormal_fit_rows)

lognormal_qq_coordinates = pd.concat(qq_frames,ignore_index=True)

display(lognormal_fit_stats)

,source,aggregation,n,epsilon_shift,shape,loc,scale,ks_stat,ks_pvalue
0,empirical,last_year,2375,0.49,0.931994,0.0,3.493498,0.105518,1.765520e-23
1,simulated,last_year,10000,0.49,1.090113,0.0,3.829766,0.029635,4.600679e-08
2,empirical,last_four,2333,0.49,0.996309,0.0,15.408227,0.062281,2.610144e-08
3,simulated,last_four,10000,0.49,0.995764,0.0,16.952071,0.018363,2.325802e-03
4,empirical,full_career,681,0.49,0.751511,0.0,116.582465,0.056957,2.316503e-02
5,simulated,full_career,10000,0.49,0.525712,0.0,129.466650,0.008277,4.972063e-01


In [27]:

rank_rows = []

for source, matrix in [("empirical", emp_trajectories),("simulated", sim_trajectories)]:
    baseline_rank = pd.Series(matrix[0]).rank(method="average").to_numpy()

    for year in YEARS:
        year_rank = pd.Series(matrix[year]).rank(method="average").to_numpy()

        correlation = np.corrcoef(baseline_rank,year_rank)[0, 1]

        rank_rows.append({"source": source,
            "career_age": int(year),
            "rank_correlation_with_year0": float(correlation)})

rank_correlations = pd.DataFrame(rank_rows)

display(rank_correlations.head())

,source,career_age,rank_correlation_with_year0
0,empirical,0,1.000000
1,empirical,1,0.253999
2,empirical,2,0.353624
3,empirical,3,0.272937
4,empirical,4,0.259772


In [28]:
wasserstein_rows = []

for year in YEARS:
    emp_values = emp_trajectories[year]
    sim_values = sim_trajectories[year]

    wasserstein_rows.append({"career_age": int(year),
        "W1_raw": float(wasserstein_distance(emp_values,sim_values)),
        "W1_log1p": float(wasserstein_distance(np.log1p(emp_values),np.log1p(sim_values)))})

wasserstein_by_year = pd.DataFrame(wasserstein_rows)

display(wasserstein_by_year.head())

,career_age,W1_raw,W1_log1p
0,0,0.822752,0.235876
1,1,1.704154,0.314654
2,2,2.122179,0.242976
3,3,1.909219,0.244308
4,4,2.298671,0.289936


In [29]:
def year_value(table, source, year, column):
    return float(table.loc[table["source"].eq(source)& table["career_age"].eq(year),column].iloc[0])


summary = {}

for source in ["empirical", "simulated"]:
    for year in [0, 5, 10, 20]:
        summary[f"{source}_mean_y{year}"] = year_value(yearwise_productivity_stats,source,year,"mean_prod")
        summary[f"{source}_median_y{year}"] = year_value(yearwise_productivity_stats,source,year,"median_prod")

summary.update({"simulated_y20_q95": float(np.quantile(sim_trajectories[20], 0.95)),
    "simulated_y20_q99": float(np.quantile(sim_trajectories[20], 0.99)),
    "simulated_y20_max": float(sim_trajectories[20].max()),
    "ks_year_max_stat": float(ks_year_max.statistic),
    "ks_year_max_pvalue": float(ks_year_max.pvalue),
    "ks_cum_y5_stat": float(ks_cum_y5.statistic),
    "ks_cum_y5_pvalue": float(ks_cum_y5.pvalue)})

diagnostic_summary = pd.DataFrame([summary])

display(diagnostic_summary.T)

,0
empirical_mean_y0,4.318336
empirical_median_y0,3.225469
empirical_mean_y5,7.754938
empirical_median_y5,6.352937
empirical_mean_y10,6.434382
empirical_median_y10,4.630756
empirical_mean_y20,5.827097
empirical_median_y20,3.750319
simulated_mean_y0,4.332366
simulated_median_y0,3.028651


In [30]:
outputs = {CANONICAL_PATH: canonical_trajectory_stats,
    YEARWISE_PATH: yearwise_productivity_stats,
    LOG_DELTA_PATH: yearwise_log_delta_stats,
    YEAR_MAX_PATH: year_of_maximum,
    CUM_Y5_PATH: cumulative_through_year5,
    KS_PATH: ks_diagnostics,
    LAPLACE_PATH: stage_raw_increment_laplace,
    LAPLACE_VALUES_PATH: stage_raw_increment_values,
    AGGREGATE_PATH: aggregate_productivity_values,
    LOGNORMAL_FIT_PATH: lognormal_fit_stats,
    QQ_PATH: lognormal_qq_coordinates,
    RANK_CORR_PATH: rank_correlations,
    WASSERSTEIN_PATH: wasserstein_by_year,
    SUMMARY_PATH: diagnostic_summary}

for path, table in outputs.items():
    table.to_csv(path, index=False)

print("Saved diagnostic outputs:")
for path in [*outputs, RANK_CHANGE_PATH]:
    print(f"  {path}")

Saved diagnostic outputs:
  /Users/samlunemagid/Desktop/mean_reversion_repo/dx/output/canonical_trajectory_stats.csv
  /Users/samlunemagid/Desktop/mean_reversion_repo/dx/output/yearwise_productivity_stats.csv
  /Users/samlunemagid/Desktop/mean_reversion_repo/dx/output/yearwise_log_delta_stats.csv
  /Users/samlunemagid/Desktop/mean_reversion_repo/dx/output/year_of_maximum.csv
  /Users/samlunemagid/Desktop/mean_reversion_repo/dx/output/cumulative_through_year5.csv
  /Users/samlunemagid/Desktop/mean_reversion_repo/dx/output/ks_diagnostics.csv
  /Users/samlunemagid/Desktop/mean_reversion_repo/dx/output/stage_raw_increment_laplace.csv
  /Users/samlunemagid/Desktop/mean_reversion_repo/dx/output/stage_raw_increment_values.csv
  /Users/samlunemagid/Desktop/mean_reversion_repo/dx/output/aggregate_productivity_values.csv
  /Users/samlunemagid/Desktop/mean_reversion_repo/dx/output/lognormal_fit_stats.csv
  /Users/samlunemagid/Desktop/mean_reversion_repo/dx/output/lognormal_qq_coordinates.csv
  /U